# The AI Chef That Can't Poison Anyone 🍳

*In-memory RDF reasoning for Neo4j — the full demo, runnable top to bottom.*

Your AI sous-chef proposed swapping margarine for **butter** in the béchamel. Fluent, plausible, delicious — and it silently turns your *vegan* lasagna into a mislabeled product with an **undeclared allergen** (EU 1169: that's a recall, not a bad review).

**LLMs guess. Ontologies know.** This notebook builds the guardrail: scope with Cypher, shape with a template, reason with Jena, validate with SHACL — `project → reason → drop`, the GDS mental model applied to meaning.

**Prerequisites**
```bash
mvn -pl n20s-server -am package          # build the sidecar
PORT=7475 java -jar n20s-server/target/n20s-server-*.jar
pip install -e n20s-python python-dotenv # the Python client (+ .env support)
```
Optionally create a sibling `.env` (see [`.env.example`](.env.example)) with `NEO4J_URI` / `NEO4J_USER` / `NEO4J_PASSWORD` / `NEO4J_DATABASE` to fetch rows from a live Neo4j over Bolt; without it, the notebook uses the same rows inline — the reasoning is identical.

In [ ]:
import json, os
from n20s import N20s

try:  # sibling .env: NEO4J_URI / NEO4J_USER / NEO4J_PASSWORD / NEO4J_DATABASE / N20S_URL
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

driver = None
if os.environ.get('NEO4J_URI'):
    from neo4j import GraphDatabase
    driver = GraphDatabase.driver(
        os.environ['NEO4J_URI'],
        auth=(os.environ.get('NEO4J_USER', 'neo4j'), os.environ.get('NEO4J_PASSWORD', '')))

n20s = N20s(os.environ.get('N20S_URL', 'http://localhost:7475'), driver=driver,
            database=os.environ.get('NEO4J_DATABASE'))
n20s.version()

## The cast

An ordinary recipe graph — nothing semantic about it. In Neo4j it would be:

```cypher
CREATE (lasagna:Recipe {id:'lasagna', name:'Lasagna', diet:'vegan', declared:['gluten']}),
       (bechamel:Recipe:Sauce {id:'bechamel', name:'Béchamel'}),
       (butter:Ingredient:Dairy {id:'butter', name:'Butter', allergens:['milk']}),
       (flour:Ingredient {id:'flour', name:'Wheat flour', allergens:['gluten']}),
       (tomato:Ingredient:Vegetable {id:'tomato', name:'Tomato', allergens:[]}),
       (lasagna)-[:CONTAINS]->(bechamel), (lasagna)-[:CONTAINS]->(tomato),
       (bechamel)-[:CONTAINS]->(butter),  (bechamel)-[:CONTAINS]->(flour);
```

Spot the two bugs the agent introduced: the lasagna still **claims vegan**, and its label still **declares only gluten**.

## Act I — Templates: the mapping is data, not code

A declarative template (MarkLogic-TDE / R2RML style) turns graph entities into triples. The **label map** is a reviewable alignment table (labels → ontology classes; unmapped labels drop out). **Lists fan out** — `allergens: ['milk']` emits one triple per element. **Missing data skips quietly** — béchamel has no `diet`, so that pattern just doesn't fire.

In [2]:
FOOD = 'http://example.org/food#'

food_template = {
    'subject': FOOD + '{id}',
    'triples': [
        {'predicate': 'http://www.w3.org/1999/02/22-rdf-syntax-ns#type',
         'object': {'from': '_labels', 'map': {
             'Ingredient': FOOD + 'Ingredient',
             'Dairy':      FOOD + 'Dairy',
             'Vegetable':  FOOD + 'Vegetable',
             'Recipe':     FOOD + 'Recipe',
             'Sauce':      FOOD + 'Sauce'}},
         'kind': 'iri'},
        {'predicate': FOOD + 'hasAllergen',
         'object': FOOD + 'allergen_{allergens}', 'kind': 'iri'},
        {'predicate': FOOD + 'claimsDiet',
         'object': FOOD + 'diet_{diet}', 'kind': 'iri'},
        {'predicate': FOOD + 'declaresAllergen',
         'object': FOOD + 'allergen_{declared}', 'kind': 'iri'},
        {'predicate': 'http://www.w3.org/2000/01/rdf-schema#label',
         'object': '{name}'},
    ],
}

Now **scope and project** — Cypher's moment. The proposal touches one dish, so we project the *lasagna's subtree* (`-[:CONTAINS*0..]->`), not the menu, into a session-scoped check graph.

In [3]:
G = 'check_lasagna'
n20s.graph.drop(G)

node_rows = [
    {'id': 'lasagna',  'name': 'Lasagna',     '_labels': ['Recipe'],
     'diet': 'vegan',  'declared': ['gluten']},
    {'id': 'bechamel', 'name': 'Bechamel',    '_labels': ['Recipe', 'Sauce']},
    {'id': 'butter',   'name': 'Butter',      '_labels': ['Ingredient', 'Dairy'],
     'allergens': ['milk']},
    {'id': 'flour',    'name': 'Wheat flour', '_labels': ['Ingredient'],
     'allergens': ['gluten']},
    {'id': 'tomato',   'name': 'Tomato',      '_labels': ['Ingredient', 'Vegetable'],
     'allergens': []},
]

if driver:  # the client runs the scoping Cypher over Bolt and converts the nodes
    result = n20s.graph.projectTemplate(
        G, food_template,
        cypher='MATCH (:Recipe {id:$id})-[:CONTAINS*0..]->(n) RETURN n',
        params={'id': 'lasagna'})
else:
    result = n20s.graph.projectTemplate(G, food_template, rows=node_rows)
result

{'graphName': 'check_lasagna',
 'rows': 5,
 'tripleCount': 17,
 'status': 'projected'}

## Act II — Edges become triples too

A second template maps **relationship types → predicates** — the second alignment table, symmetric with labels → classes. Entities are addressed by **name** (`{s}`, `{r}`, `{t}`), so a template can never silently reverse an edge.

In [4]:
contains_template = {
    'subject': FOOD + '{s.id}',
    'triples': [
        {'predicate': {'from': 'r._type', 'map': {'CONTAINS': FOOD + 'contains'}},
         'object': FOOD + '{t.id}', 'kind': 'iri'},
    ],
}

hop_rows = [
    {'s': {'id': 'lasagna'},  'r': {'_type': 'CONTAINS'}, 't': {'id': 'bechamel'}},
    {'s': {'id': 'lasagna'},  'r': {'_type': 'CONTAINS'}, 't': {'id': 'tomato'}},
    {'s': {'id': 'bechamel'}, 'r': {'_type': 'CONTAINS'}, 't': {'id': 'butter'}},
    {'s': {'id': 'bechamel'}, 'r': {'_type': 'CONTAINS'}, 't': {'id': 'flour'}},
]

if driver:  # Cypher's * walks the tree; multi-column records become named rows
    result = n20s.graph.projectTemplate(
        G, contains_template,
        cypher='''MATCH p = (:Recipe {id:$id})-[:CONTAINS*]->()
                  UNWIND relationships(p) AS r
                  RETURN DISTINCT startNode(r) AS s, r, endNode(r) AS t''',
        params={'id': 'lasagna'}, ifExists='append')
else:
    result = n20s.graph.projectTemplate(G, contains_template, rows=hop_rows,
                                        ifExists='append')
result

{'graphName': 'check_lasagna',
 'rows': 4,
 'tripleCount': 4,
 'status': 'projected'}

## Act III — Two triples of knowledge

The entire ontology this demo needs: *dairy things are animal products*, and *containment is transitive*.

In [5]:
ontology = '''
@prefix food: <http://example.org/food#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix owl:  <http://www.w3.org/2002/07/owl#> .
food:Dairy    rdfs:subClassOf food:AnimalProduct .
food:contains a owl:TransitiveProperty .
'''
n20s.graph.addTurtle(G, ontology)

{'graphName': 'check_lasagna',
 'triplesBefore': 21,
 'triplesAfter': 23,
 'added': 2}

One domain rule — **allergens propagate up containment** — and we ask with **backward chaining**: the rule recurses *during the query*, nothing is materialized. Watch the milk arrive from two hops down (*lasagna → béchamel → butter*).

In [6]:
propagation_rule = (
    '[allergenPropagation: (?r http://example.org/food#contains ?i)'
    ' (?i http://example.org/food#hasAllergen ?a)'
    ' -> (?r http://example.org/food#hasAllergen ?a)]')

n20s.graph.queryWithRules(
    G,
    'SELECT ?a WHERE { <http://example.org/food#lasagna>'
    ' <http://example.org/food#hasAllergen> ?a }',
    propagation_rule)

[{'a': 'http://example.org/food#allergen_gluten'},
 {'a': 'http://example.org/food#allergen_milk'}]

And what should a vegan fear? `AnimalProduct` **exists nowhere in the property graph** — no label, no property. It exists in one ontology triple, and the OWL profile applies it at query time.

In [7]:
n20s.graph.query(
    G,
    'SELECT ?x WHERE { <http://example.org/food#lasagna>'
    ' <http://example.org/food#contains> ?x .'
    ' ?x a <http://example.org/food#AnimalProduct> }',
    profile='OWL_MICRO')

[{'x': 'http://example.org/food#butter'}]

The receipt — after both questions, the graph still holds **exactly what was projected**. The answers were computed, returned, and gone.

In [8]:
n20s.graph.list()

[{'graphName': 'check_lasagna', 'tripleCount': 23}]

## Act IV — SHACL renders the verdict

Shapes validate *asserted* triples, so this is where **forward chaining** earns its keep: materialize the entailments once, then validate. (Backward for one-shot questions, forward when shapes need the entailments to exist — that's the rule of thumb.)

In [9]:
n20s.graph.inferWithRules(G, propagation_rule, profile='OWL_MICRO')

{'graphName': 'check_lasagna',
 'triplesBefore': 23,
 'triplesAfter': 155,
 'newTriples': 132,
 'profile': 'CUSTOM (1 rules)'}

In [10]:
shapes = '''
@prefix sh:   <http://www.w3.org/ns/shacl#> .
@prefix food: <http://example.org/food#> .
food:VeganShape a sh:NodeShape ;
    sh:targetSubjectsOf food:claimsDiet ;
    sh:sparql [
        sh:message "Claims vegan but contains an animal product" ;
        sh:select "PREFIX food: <http://example.org/food#> SELECT $this ?value WHERE { $this food:claimsDiet food:diet_vegan . $this food:contains ?value . ?value a food:AnimalProduct . }" ;
    ] .
food:DeclarationShape a sh:NodeShape ;
    sh:targetSubjectsOf food:declaresAllergen ;
    sh:sparql [
        sh:message "EU 1169: allergen present but not declared on the label" ;
        sh:select "PREFIX food: <http://example.org/food#> SELECT $this ?value WHERE { $this food:hasAllergen ?value . FILTER NOT EXISTS { $this food:declaresAllergen ?value } }" ;
    ] .
'''
n20s.graph.addTurtle(G, shapes)

for v in n20s.graph.validate(G):
    print(f"[{v['severity']}] {v['focusNode'].split('#')[-1]}: {v['message']}")
    print(f"            → {v['value'].split('#')[-1]}")

[Violation] lasagna: EU 1169: allergen present but not declared on the label
            → allergen_milk
[Violation] lasagna: Claims vegan but contains an animal product
            → butter


Both bugs the agent introduced, caught — the vegan lie by ontology-driven inference, the undeclared milk by regulation-shaped constraint. The agent can now regenerate with these violations in context: margarine stays, or the label changes. **Nothing ships that the rules forbid.**

## Drop — the check was ephemeral

The graph of record never changed. Every step above is replayable in front of an auditor.

In [11]:
n20s.graph.drop(G)

{'graphName': 'check_lasagna', 'status': 'dropped'}

---

**Learn more**

- Repo: [github.com/halftermeyer/neo4j-n20s](https://github.com/halftermeyer/neo4j-n20s) — plugin + server + Python client
- Template semantics, every rule as *row + template → triples*: [TEMPLATES.md](https://github.com/halftermeyer/neo4j-n20s/blob/main/TEMPLATES.md)
- Agent integration guide: [BEST_PRACTICES.md](https://github.com/halftermeyer/neo4j-n20s/blob/main/BEST_PRACTICES.md)

*Scope with Cypher. Reason with ontologies. Verify everything.*